# OSR-Bench evaluation framework — adaptation to GQA-CoT

**Paper:** Dongfang et al., *Are Multimodal Large Language Models Ready for Omnidirectional Spatial Reasoning?* (CVPR Findings 2026), pp. 9759–9769.

## 0. Feasibility

**It does not fit.** The paper evaluates 8 MLLMs, including GPT-4o and Gemini-1.5-Pro (API-only) and open-source checkpoints at 72B/78B/90B (Sec. 3.3). Qwen2.5-VL-72B in fp16 alone needs ~144 GB of weights; this session has 2×T4 = 32 GB. The paper's data (4,100 360° panoramas from 1,526 scenes, ~153k QA pairs, Sec. 3.1) is also not the data mounted here. What follows is a **faithful reduced adaptation**: the paper's two-stage evaluation *protocol* (Sec. 3.2) run on the mounted GQA-CoT val file with a small same-family checkpoint.

### Deviation table

| Item | Paper's value | This run | Effect on results |
|---|---|---|---|
| Benchmark data | OSR-Bench, 4,100 panoramas, 1,526 scenes, ~153k QA (Sec. 3.1) | GQA-CoT val, 5,422 pinhole images, 9,855 records | Different task and imagery; scores are **not** comparable to Tables 2–5 |
| Ground-truth objects | 3D layout annotations from ReplicaPano + DeepPanoContext, all instances per scene (Sec. 3.1) | `bboxs` of the annotated referents only (~1.8 records/image) | GT pools are *sparse*. Precision and CHAIR are biased: any correctly-named but unannotated object counts as hallucinated. Precision is a lower bound, CHAIR an upper bound. |
| Cognitive map | Top-down 10×10 grid from 3D centroids, Algorithm 1 | 10×10 grid over the **image plane** from 2D bbox centroids | Different geometry; the map is not a top-down room layout |
| Object classes | 65 categories from layout annotations (Sec. 3.1) | 928 distinct answer strings used as class labels | Open vocabulary vs. a closed 65-class set; a predicted synonym scores as a miss |
| Reasoning categories | object counting, relative distance, relative direction (Sec. 3.1) | spatial-relation subset + other; **no real counting questions exist** in this file | The paper's object-counting category survives only as synthetic negative-sampling questions. Relative distance/direction as the paper defines them do not exist in GQA. |
| Models | 8 MLLMs, up to 90B, incl. proprietary (Sec. 3.3, Table 2) | Qwen2.5-VL-3B-Instruct | Same family as the paper's Qwen2.5-VL-72B, 24× smaller; expect substantially lower scores across every metric |
| Evaluator | regex rule-based **and** LLM-based judge (Sec. 3.2) | rule-based only | Answers that are semantically right but phrased differently score 0 more often |
| Negative sampling | Popular Sampling **and** Adversarial Sampling (Sec. 3.1, Appendix A.2) | Popular Sampling only | Appendix A.2 is not in the provided pages; adversarial variant not implemented |
| Eval size | ~153k QA pairs | `CFG.MAX_IMAGES` images and their records (see `results.csv` → `n_eval`) | A subset score is not the paper's score |
| Decoding | greedy (Sec. 3.3) | greedy | matched |

## 1. Goal and architecture (3–5 sentences)

The paper asks whether MLLMs can do spatial reasoning on 180°×360° imagery, and answers it by building OSR-Bench rather than by proposing a trained model. Ground-truth "omni-cognitive maps" are extracted from 3D layout annotations by binning object centroids into a 10×10 grid (Algorithm 1), and those maps both serve as evaluation references and drive templated QA generation over three categories: object counting, relative distance, relative direction (Sec. 3.1). A negative-sampling stage injects objects that do not exist in the scene into prompts and QA templates to probe hallucination (Sec. 3.1). Evaluation is a two-round visual dialogue: stage 1 scores the model's generated cognitive map against ground truth with Hungarian matching and a rotation-invariant F1, stage 2 scores answers with regex rules and an LLM judge (Sec. 3.2). There is **no trainable component** — all evaluation is zero-shot with greedy decoding (Sec. 3.3).

## 2. Module table

| Module | Function | Input | Output | Section |
|---|---|---|---|---|
| Omni-cognitive map extraction | Bin object centroids into a G×G grid, G=10 | Scene annotations S | Grid `C` (10×10, each cell a set of classes) + class-count dict `D` | Algorithm 1 |
| Object pool | Collect the extracted instances per scene | Same objects as above | Set of class names | Sec. 3.1, Fig. 3 |
| Negative sampling | Inject classes absent from the scene | Object pool + dataset class stats | Augmented category list / QA templates | Sec. 3.1 |
| QA template generation | Fill predefined templates from map + pool | Map, pool | QA pairs with verifiable answers | Sec. 3.1, Appendix A.1 |
| Stage-1 evaluator | Per-class Hungarian assignment on Euclidean distance, threshold 2.0 grid units, max over rotations | Predicted map `P`, GT map `G` | Avg dist, Precision, Recall, F1, success rate | Eqs. 1–4, Table 2 |
| CHAIR | Hallucinated classes/instances among predictions | Predicted map, GT pool | CHAIR_S, CHAIR_I, CHAIR_instance | Sec. 3.2, Table 4 |
| Stage-2 evaluator | Regex match; percentage score for counting, binary for distance/direction | Model answer, GT answer | Per-category score | Sec. 3.2, Table 3 |

Input/output *shapes*: the paper specifies only the grid, `G = 10 × 10` (Algorithm 1), and the matrices `D^i ∈ R^{|G_i|×|P_i|}`, `X^i ∈ {0,1}^{|G_i|×|P_i|}` (Eqs. 1–3). Tensor shapes for the MLLMs are **NOT SPECIFIED**.

## 3. Data flow

1. Panoramic image + 3D layout annotation → extract object instances (Sec. 3.1).
2. Instances → GT omni-cognitive map (Algorithm 1) **and** object pool (Fig. 3).
3. Pool → negative sampling injects non-existent classes → two dataset versions, clean and negative (Sec. 3.1).
4. Pool + map + templates → QA pairs (Appendix A.1).
5. Round 1: image + category prompt → model generates a cognitive map → scored by Eqs. 1–4 and CHAIR (Sec. 3.2).
6. Round 2: image + generated map + question → model answers → regex / LLM scoring (Sec. 3.2).
7. Ablation: the same questions answered **without** the cognitive-map round (Table 3, "w/o cogmap").

## 4. Reproduction table

| Item | Value | Reference |
|---|---|---|
| Dataset | OSR-Bench, built on ReplicaPano and DeepPanoContext (iGibson) | Sec. 3.1 |
| Dataset size | 4,100 images, 1,526 scenes, 65 categories, ~153k QA | Sec. 3.1, Table 1 |
| Split | NOT SPECIFIED (no train/val/test split is described; all evaluation is zero-shot) | Sec. 3.3 |
| Preprocessing | Grid mapping `i ← ⌊(x−x_min)/(x_max−x_min)·G⌋`, likewise `j`; G=10 | Algorithm 1 |
| Image preprocessing (resize, normalization) | NOT SPECIFIED | — |
| Feature selection | Not applicable — no feature-selection stage exists | — |
| Model per stage | Stage 1 and stage 2 both use the same MLLM; 8 evaluated (GPT-4o, Gemini-1.5-Pro, Llama-3.2-90B, Qwen2.5-VL-72B, InternVL2_5-78B, LLaVA-v1.5-13B, deepseek-vl2, Janus-Pro-7B) | Sec. 3.3, Table 2 |
| Loss | None — zero-shot, no task-specific fine-tuning | Sec. 3.3 |
| Optimizer | None — no training | Sec. 3.3 |
| Learning rate / schedule | None — no training | Sec. 3.3 |
| Batch size | NOT SPECIFIED | — |
| Epochs | None — no training | Sec. 3.3 |
| Regularization | None — no training | Sec. 3.3 |
| Decoding | Greedy, all models | Sec. 3.3 |
| Matching threshold | 2.0 grid units | Sec. 3.2 |
| Rotations searched | "under rotational transformations… select the one that yields the highest F1"; the *set* of rotations is NOT SPECIFIED | Sec. 3.2 |
| Metrics | Avg dist, Precision, Recall, F1, success rate; CHAIR_S / CHAIR_I / CHAIR_instance; per-category QA score | Sec. 3.2, Tables 2 and 4 |
| Counting score | "100% when the model's count exactly matches… decreasing proportionally"; exact formula NOT SPECIFIED | Sec. 3.2 |
| Distance/direction score | Binary correct/incorrect | Sec. 3.2 |
| Ablations | w/ vs. w/o cognitive map (Table 3); clean vs. negative sampling (Table 4); thinking-mode prompt, Gemini only (Table 5) | Sec. 3.3.1 |
| Prompts | Cognitive-map prompt, CoT template, LLM-judge prompts are in Appendices C.1–C.3, which are not in the provided pages → NOT SPECIFIED | Sec. 3.2 |

## 5. Assumptions table

Every row here is my choice, not the paper's. Each is marked in the code with `# ASSUMPTION:`.

| # | Missing detail | Value used | Why |
|---|---|---|---|
| A1 | Set of rotations for the rotation-invariant F1 (Sec. 3.2 says only "rotational transformations") | 0°, 90°, 180°, 270° about the grid centre | The natural set for a square grid |
| A2 | Counting-score formula ("decreasing proportionally", Sec. 3.2) | `max(0, 1 − |pred − gt| / max(gt, 1))` | Simplest function that is 1.0 at equality and decreases linearly |
| A3 | Cognitive-map prompt text (Appendix C.1, not provided) | JSON-emitting prompt listing the candidate categories, mirroring the format in Fig. 5 | Fig. 5 shows the model reasoning over "the provided categories are {…}" and a JSON map |
| A4 | Output format for answers | Free text; `<answer>…</answer>` honoured if present | Fig. 5 shows that tag format |
| A5 | Popular-sampling definition (Appendix A.2, not provided) | The K most frequent dataset classes absent from this image's GT pool | POPE-style "popular" sampling is frequency-based; the paper names but does not define it |
| A6 | Object class labels per box | The record's `answer` string labels its box | Verified on the mounted file: every record has exactly one box and all 9,855 reasoning traces end in `query name`, `choose name` or `choose <comparative>`, so the answer names the boxed object. Yields 1.47 classes/image, no empty images. |
| A7 | Full-frame box threshold | box area ≥ 0.80 of the image | Your setup asks for it; the paper has no such notion |
| A8 | Grounding IoU threshold | 0.5 | Your setup asks for grounding accuracy; the paper has no bbox-grounding task |
| A9 | Number of images evaluated | `CFG.MAX_IMAGES`, sampled with `CFG.SEED` | 12-hour session limit; see deviation table |
| A10 | Image token budget | `max_pixels = 1024·28·28` | Fits T4 memory; the paper does not specify image preprocessing |
| A11 | Predicted maps that ignore the grid format | Coordinates outside [0,G) are read as pixels and mapped through Algorithm 1 | The model answers in pixel coordinates. Strict adherence is reported separately as `succ_rate_strict_grid`, the honest analogue of Table 2's Succ. Rate. |
| A12 | Ablation pairing | w/ and w/o cogmap scored on the identical record set | An unpaired comparison confounds the ablation with which images produced a parsable map |

**No results are reported in this document.** Every number appears only after you run the notebook, in `/kaggle/working/results.csv`.

## CONFIG

In [ ]:
# ============================== CONFIG ==============================
INSTALL_DEPS = True  # set False if the wheels are already present

if INSTALL_DEPS:
    import subprocess, sys as _sys
    subprocess.run(
        [_sys.executable, "-m", "pip", "install", "-q",
         "transformers==4.51.3", "accelerate==1.6.0", "safetensors==0.5.3"],
        check=True,
    )

import os, sys, json, re, math, time, random, gc, platform, traceback
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy.optimize import linear_sum_assignment

Image.MAX_IMAGE_PIXELS = None


class CFG:
    # ---- paths (from the task setup) ----
    ANN_PATH = ("/kaggle/input/notebooks/khoangoo/test-dataset-visual-cot/"
                "visual-cot/cot_with_detailed_reasoning_steps/gqa_cot_val.jsonl")
    IMAGE_ROOT = "/kaggle/input/datasets/lyte69/gqa-images/images"
    OUT_DIR = "/kaggle/working"

    EXPECTED_RECORDS = 9855
    EXPECTED_IMAGES = 5422

    # ---- model ----
    # Paper Sec. 3.3 evaluates Qwen2.5-VL-72B-Instruct. 72B does not fit on 2x T4.
    # Same family, smaller checkpoint. Logged in the deviation table.
    MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
    MODEL_SHORT = "Qwen2.5-VL-3B-Instruct"
    # ASSUMPTION: paper does not specify image preprocessing; using a 1024-patch budget
    MIN_PIXELS = 256 * 28 * 28
    MAX_PIXELS = 1024 * 28 * 28

    # ---- paper-specified constants ----
    GRID_G = 10            # Algorithm 1: "Grid size G = 10 x 10"
    MATCH_THRESHOLD = 2.0  # Sec. 3.2: "a distance threshold of 2.0 grid units"
    GREEDY = True          # Sec. 3.3: "we employ greedy decoding across all models"

    # ---- assumptions (see assumptions table) ----
    ROTATIONS = (0, 1, 2, 3)      # A1: k * 90 degrees
    N_NEGATIVE_CLASSES = 3        # A5: injected absent classes per image
    FULL_FRAME_AREA_FRAC = 0.80   # A7
    IOU_THRESHOLD = 0.5           # A8
    MAX_IMAGES = 250              # A9: raise for a longer run; 0 means "all"
    SEED = 0

    # ---- generation budgets ----
    MAX_NEW_TOKENS_COGMAP = 512
    MAX_NEW_TOKENS_QA = 96
    MAX_NEW_TOKENS_BOX = 64

    # ---- stage switches ----
    RUN_STAGE1_CLEAN = True
    RUN_STAGE1_NEGATIVE = True
    RUN_STAGE2_WITH_COGMAP = True
    RUN_STAGE2_WITHOUT_COGMAP = True
    RUN_STAGE2_NEGATIVE = True
    RUN_GROUNDING = True


os.makedirs(CFG.OUT_DIR, exist_ok=True)

random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
torch.manual_seed(CFG.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.SEED)

print("python           :", platform.python_version())
print("torch            :", torch.__version__)
print("cuda available   :", torch.cuda.is_available())
print("gpu count        :", torch.cuda.device_count())
for _i in range(torch.cuda.device_count()):
    _p = torch.cuda.get_device_properties(_i)
    print(f"  gpu[{_i}]        : {_p.name}, {_p.total_memory / 1024**3:.1f} GiB")
print()
print("NONDETERMINISM: greedy decoding fixes the sampling path, but cuBLAS/cuDNN")
print("kernel selection, the number of visible GPUs, and the exact transformers /")
print("torch build can still shift generated text. Weight download resolves to the")
print("current HF revision of the checkpoint, which is not pinned here.")

## CELL 1 — SETUP CHECKS

Prints the mounts, resolves every unique image, drops records whose stored `width`/`height` disagree with the file on disk, clamps out-of-range boxes, and flags full-frame boxes. Raises and stops on anything unexpected. Never substitutes an image.

In [ ]:
# ============================== CELL 1: SETUP CHECKS ==============================
print("ANN_PATH   :", CFG.ANN_PATH)
print("IMAGE_ROOT :", CFG.IMAGE_ROOT)

if not os.path.exists(CFG.ANN_PATH):
    print("\n--- what is actually mounted under /kaggle/input ---")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
        if depth > 4:
            dirs[:] = []
            continue
        print(root, f"({len(files)} files)")
    raise FileNotFoundError(f"Annotation file not found: {CFG.ANN_PATH}")

if not os.path.isdir(CFG.IMAGE_ROOT):
    print("\n--- what is actually mounted under /kaggle/input ---")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
        if depth > 4:
            dirs[:] = []
            continue
        print(root, f"({len(files)} files)")
    raise FileNotFoundError(f"IMAGE_ROOT not found: {CFG.IMAGE_ROOT}")

image_files = os.listdir(CFG.IMAGE_ROOT)
print("files in IMAGE_ROOT :", len(image_files))

# ---- load annotations ----
raw_records = []
with open(CFG.ANN_PATH, "r") as f:
    for line_no, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            raw_records.append(json.loads(line))
        except json.JSONDecodeError as e:
            raise ValueError(f"Malformed JSON on line {line_no} of {CFG.ANN_PATH}: {e}")

print("records loaded      :", len(raw_records))
if len(raw_records) != CFG.EXPECTED_RECORDS:
    print(f"WARNING: expected {CFG.EXPECTED_RECORDS} records, found {len(raw_records)}")

required_fields = ["question", "answer", "full_answer", "image", "width", "height", "bboxs"]
missing_field_rows = [i for i, r in enumerate(raw_records)
                      if any(k not in r for k in required_fields)]
if missing_field_rows:
    raise KeyError(f"{len(missing_field_rows)} records are missing required fields; "
                   f"first offenders: {missing_field_rows[:20]}")

unique_images = sorted({r["image"] for r in raw_records})
print("unique images       :", len(unique_images))
if len(unique_images) != CFG.EXPECTED_IMAGES:
    print(f"WARNING: expected {CFG.EXPECTED_IMAGES} unique images, found {len(unique_images)}")
print(f"records per image   : {len(raw_records) / max(len(unique_images), 1):.2f}")

# ---- resolve every unique image and read its real size ----
t0 = time.time()
real_size = {}
missing = []
unreadable = []
for fn in unique_images:
    p = os.path.join(CFG.IMAGE_ROOT, fn)
    if not os.path.exists(p):
        missing.append(fn)
        continue
    try:
        with Image.open(p) as im:
            real_size[fn] = im.size  # (width, height); header read only
    except Exception as e:
        unreadable.append((fn, repr(e)))

print(f"resolved            : {len(real_size)} / {len(unique_images)} "
      f"({time.time() - t0:.1f}s)")

if missing:
    print(f"MISSING {len(missing)} images. First 20 filenames:")
    for fn in missing[:20]:
        print("  ", fn)
    raise FileNotFoundError(
        f"{len(missing)} of {len(unique_images)} images do not resolve under "
        f"{CFG.IMAGE_ROOT}. This is a path problem, not a missing dataset. "
        "Fix IMAGE_ROOT and re-run. No substitute images are generated."
    )

if unreadable:
    print(f"UNREADABLE {len(unreadable)} images. First 20:")
    for fn, err in unreadable[:20]:
        print("  ", fn, err)
    raise OSError(f"{len(unreadable)} images could not be opened.")

# ---- size agreement: bboxs are pixel coordinates, so a resized image invalidates them ----
records = []
n_size_mismatch = 0
for r in raw_records:
    w_real, h_real = real_size[r["image"]]
    if int(r["width"]) != int(w_real) or int(r["height"]) != int(h_real):
        n_size_mismatch += 1
        continue
    records.append(r)

print("dropped (size mismatch):", n_size_mismatch)
if len(records) == 0:
    raise RuntimeError("Every record disagreed with its image on disk. The images have "
                       "been re-encoded or resized; the boxes cannot be trusted.")
frac_dropped = n_size_mismatch / len(raw_records)
if frac_dropped > 0.05:
    print(f"WARNING: {frac_dropped:.1%} of records dropped. Many mismatches means the "
          "images were re-encoded and the pixel boxes cannot be trusted.")

# ---- clamp boxes, flag full-frame boxes, drop degenerate boxes ----
n_clamped = 0
n_degenerate = 0
n_fullframe_boxes = 0
n_records_with_fullframe = 0

for r in records:
    W, H = int(r["width"]), int(r["height"])
    kept, was_clamped, has_full = [], False, False
    for b in r["bboxs"]:
        x1, y1, x2, y2 = [float(v) for v in b[:4]]
        cx1, cy1 = min(max(x1, 0.0), W), min(max(y1, 0.0), H)
        cx2, cy2 = min(max(x2, 0.0), W), min(max(y2, 0.0), H)
        if (cx1, cy1, cx2, cy2) != (x1, y1, x2, y2):
            was_clamped = True
        if cx2 <= cx1 or cy2 <= cy1:
            n_degenerate += 1
            continue
        area_frac = ((cx2 - cx1) * (cy2 - cy1)) / float(W * H)
        if area_frac >= CFG.FULL_FRAME_AREA_FRAC:  # A7
            has_full = True
            n_fullframe_boxes += 1
        kept.append([cx1, cy1, cx2, cy2])
    if was_clamped:
        n_clamped += 1
    r["bboxs_clamped"] = kept
    r["has_fullframe_box"] = has_full
    if has_full:
        n_records_with_fullframe += 1

records = [r for r in records if len(r["bboxs_clamped"]) > 0]

print("records with >=1 clamped box :", n_clamped)
print("degenerate boxes removed     :", n_degenerate)
print("full-frame boxes (area >= "
      f"{CFG.FULL_FRAME_AREA_FRAC:.0%})    :", n_fullframe_boxes)
print("records containing one       :", n_records_with_fullframe)
print("records surviving cell 1     :", len(records))
print()
print("Grounding accuracy is reported twice in results.csv: over all scored records, and")
print("over records with no full-frame box, because a full-frame box makes IoU trivial.")

if len(records) == 0:
    raise RuntimeError("No usable records left after validation.")

## LOAD DATA / PREPROCESSING

Builds, per image: an object pool, a ground-truth 10×10 cognitive map (Algorithm 1, applied to bbox centroids in the image plane), and a popular-sampling set of absent classes.

In [ ]:
# ============================== PREPROCESSING ==============================

# --- text normalisation used by every scorer ---
NUM_WORDS = {
    "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4", "five": "5",
    "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10",
    "eleven": "11", "twelve": "12", "no": "0", "none": "0",
}
ARTICLES = {"a", "an", "the"}


def normalize_text(s):
    # lowercase -> strip xml-ish tags -> punctuation to space -> number words to digits
    # -> drop articles -> collapse whitespace
    s = str(s).lower()
    s = re.sub(r"<[^>]{0,40}>", " ", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    toks = [NUM_WORDS.get(t, t) for t in s.split()]
    toks = [t for t in toks if t not in ARTICLES]
    return " ".join(toks)


# --- A6: class labels come from the record's answer ---
# Every record in this file carries exactly one box, and every reasoning trace ends in a
# name query, so the answer string names the object the box encloses. The previous rule
# matched question tokens against an answer-derived vocabulary, labelled only 0.37 classes
# per image and left 3,615 images with no labelled object, which forced stage-1 F1 to 0
# regardless of model quality.
def box_labels(rec):
    lab = normalize_text(rec["answer"])
    return [lab] * len(rec["bboxs_clamped"]) if lab else []


def grid_cell(cx, cy, W, H, G):
    # Algorithm 1, lines 3 and 5, with the layout bounds replaced by the image bounds.
    i = int(math.floor((cx - 0.0) / max(float(W), 1e-9) * G))
    j = int(math.floor((cy - 0.0) / max(float(H), 1e-9) * G))
    return min(max(i, 0), G - 1), min(max(j, 0), G - 1)


# --- group records by image, build GT cognitive map + object pool (Algorithm 1) ---
by_image = defaultdict(list)
for r in records:
    by_image[r["image"]].append(r)

class_frequency = Counter()
image_data = {}
for fn, recs in by_image.items():
    W, H = int(recs[0]["width"]), int(recs[0]["height"])
    cogmap = defaultdict(list)
    for r in recs:
        labels = box_labels(r)
        r["box_labels"] = labels
        for lab, b in zip(labels, r["bboxs_clamped"]):
            cx, cy = (b[0] + b[2]) / 2.0, (b[1] + b[3]) / 2.0
            cogmap[lab].append(grid_cell(cx, cy, W, H, CFG.GRID_G))
    pool = sorted(cogmap.keys())
    for c in pool:
        class_frequency[c] += 1
    image_data[fn] = {"image": fn, "width": W, "height": H,
                      "cogmap": {k: sorted(set(v)) for k, v in cogmap.items()},
                      "pool": pool, "records": recs}

pool_sizes = [len(d["pool"]) for d in image_data.values()]
mean_pool = float(np.mean(pool_sizes))
print("images with a GT cognitive map:", len(image_data))
print("mean classes per image        :", round(mean_pool, 4))
print("images with zero classes      :", int(np.sum(np.array(pool_sizes) == 0)))
print("distinct classes overall      :", len(class_frequency))
if mean_pool < 0.5:
    raise RuntimeError(f"Class labelling collapsed to {mean_pool:.2f} classes per image; "
                       "stage-1 F1 would be 0 by construction, not by measurement.")

# --- A5: popular negative sampling - frequent classes absent from this image ---
popular_ranked = [c for c, _ in class_frequency.most_common()]
for fn, d in image_data.items():
    present = set(d["pool"])
    negs = []
    for c in popular_ranked:
        if c not in present:
            negs.append(c)
        if len(negs) == CFG.N_NEGATIVE_CLASSES:
            break
    d["negative_classes"] = negs

# --- A9: subset selection ---
all_images = sorted(image_data.keys())
if CFG.MAX_IMAGES and CFG.MAX_IMAGES < len(all_images):
    rng = random.Random(CFG.SEED)
    eval_images = sorted(rng.sample(all_images, CFG.MAX_IMAGES))
else:
    eval_images = all_images
eval_records = [r for fn in eval_images for r in image_data[fn]["records"]]

print()
print("EVAL SUBSET  images :", len(eval_images), "of", len(all_images))
print("EVAL SUBSET records :", len(eval_records), "of", len(records))
print("This is a subset score, not the paper's score. It is logged as a deviation.")


# --- question routing (adaptation; the paper's three categories do not exist here) ---
COUNT_RE = re.compile(r"\b(how many|what number of|number of)\b")
SPATIAL_RE = re.compile(
    r"\b(left|right|above|below|behind|front|top|bottom|under|underneath|beside|"
    r"next to|near|closest|farthest|between|side)\b")


def question_category(q):
    qn = normalize_text(q)
    if COUNT_RE.search(qn):
        return "object_count"
    if SPATIAL_RE.search(qn):
        return "spatial_relation"
    return "other"


cat_counts = Counter(question_category(r["question"]) for r in eval_records)
print("category counts in subset:", dict(cat_counts))


# --- prompt builders (A3, A4: exact prompts are in Appendix C, not provided) ---
def cogmap_prompt(categories):
    cats = ", ".join(categories)
    return (
        "You are given an image. Build a cognitive map of the objects in it on a "
        f"{CFG.GRID_G} by {CFG.GRID_G} grid laid over the image, with row and column "
        f"indices from 0 to {CFG.GRID_G - 1}. Row 0 is the top of the image and column 0 "
        "is the left edge.\n"
        f"The provided categories are: {cats}.\n"
        "For every category that is actually visible, give the grid cell of each "
        "instance's centre. Omit categories that are not visible.\n"
        "Do NOT give pixel coordinates and do NOT give bounding boxes. Every position is "
        f"exactly two integers in 0..{CFG.GRID_G - 1}.\n"
        "Respond with JSON only, no other text, using the real category names as keys, "
        'like this worked example: {"chair": [[2, 7], [3, 7]], "lamp": [[8, 1]]}'
    )


def qa_prompt(question):
    return (question.strip() + "\nAnswer with a single word or a short phrase, "
            "wrapped in <answer></answer> tags.")


def negative_count_question(cls):
    # Fig. 2 shows exactly this template with the answer 0 under negative sampling.
    return f"How many {cls}(s) are in this image?"


def grounding_prompt(question):
    return (
        "Question: " + question.strip() + "\n"
        "Locate the single region of the image that this question is about. "
        "Respond with JSON only, no other text, in exactly this form:\n"
        '{"bbox": [x1, y1, x2, y2]}\n'
        "Coordinates are pixels in the image you were given."
    )

## MODEL

In [ ]:
# ============================== MODEL ==============================
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

if torch.cuda.is_available():
    N_GPU = torch.cuda.device_count()
    # T4 (sm_75) supports fp16 but not bf16.
    DTYPE = torch.float16
    # A 3B checkpoint in fp16 is ~6.2 GB and fits one 16 GB T4. Shard only if a larger
    # checkpoint is configured, so the notebook still runs unmodified on 1 GPU.
    DEVICE_MAP = "auto" if N_GPU > 1 and not CFG.MODEL_ID.endswith("3B-Instruct") else None
    DEVICE = "cuda:0"
else:
    N_GPU = 0
    DTYPE = torch.float32
    DEVICE_MAP = None
    DEVICE = "cpu"

print("device:", DEVICE, "| dtype:", DTYPE, "| device_map:", DEVICE_MAP, "| gpus:", N_GPU)

processor = AutoProcessor.from_pretrained(
    CFG.MODEL_ID, min_pixels=CFG.MIN_PIXELS, max_pixels=CFG.MAX_PIXELS
)

_load_kwargs = dict(torch_dtype=DTYPE, attn_implementation="sdpa", low_cpu_mem_usage=True)
if DEVICE_MAP is not None:
    _load_kwargs["device_map"] = DEVICE_MAP

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(CFG.MODEL_ID, **_load_kwargs)
if DEVICE_MAP is None:
    model = model.to(DEVICE)
model.eval()

PARAM_COUNT = int(sum(p.numel() for p in model.parameters()))
print("parameters:", PARAM_COUNT)

if torch.cuda.is_available():
    for _i in range(N_GPU):
        torch.cuda.reset_peak_memory_stats(_i)


def peak_gpu_gib():
    if not torch.cuda.is_available():
        return float("nan")
    return sum(torch.cuda.max_memory_allocated(i) for i in range(N_GPU)) / 1024 ** 3


def load_image(fn):
    p = os.path.join(CFG.IMAGE_ROOT, fn)
    with Image.open(p) as im:
        return im.convert("RGB")


def build_messages(turns, with_image=True):
    # turns: list of (role, text). The image rides on the first user turn.
    msgs = []
    for k, (role, text) in enumerate(turns):
        if k == 0 and role == "user" and with_image:
            content = [{"type": "image"}, {"type": "text", "text": text}]
        else:
            content = [{"type": "text", "text": text}]
        msgs.append({"role": role, "content": content})
    return msgs


@torch.inference_mode()
def generate(turns, pil_image, max_new_tokens):
    msgs = build_messages(turns, with_image=pil_image is not None)
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=[text],
        images=[pil_image] if pil_image is not None else None,
        return_tensors="pt",
    )
    inputs = {k: (v.to(DEVICE) if hasattr(v, "to") else v) for k, v in inputs.items()}
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=not CFG.GREEDY)
    trimmed = out[:, inputs["input_ids"].shape[1]:]
    decoded = processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
    grid_thw = inputs.get("image_grid_thw")
    if grid_thw is not None:
        g = grid_thw[0].tolist()
        # Qwen2.5-VL emits coordinates in the smart-resized image space.
        # patch size is 14, so resized H = grid_h * 14, resized W = grid_w * 14.
        resized_hw = (int(g[1]) * 14, int(g[2]) * 14)
    else:
        resized_hw = None
    return decoded, resized_hw

## TRAIN

The paper performs no training: *"All evaluations are conducted in strict zero-shot settings without any task-specific fine-tuning"* (Sec. 1, repeated in Sec. 3.3). This section therefore records zero training time and zero training steps — measured, not assumed.

In [ ]:
# ============================== TRAIN ==============================
# Sec. 3.3: zero-shot, no task-specific fine-tuning. There is no training loop to run.
train_t0 = time.time()
n_training_steps = 0
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
for p in model.parameters():
    p.requires_grad_(False)
train_time_s = time.time() - train_t0

print("training steps executed :", n_training_steps)
print(f"training wall time (s)  : {train_time_s:.4f}")
print("params left trainable   :", sum(p.numel() for p in model.parameters() if p.requires_grad))
assert n_training_steps == 0

## EVALUATE — stage 1: omni-cognitive map generation

Hungarian assignment per object class on the Euclidean distance matrix (Eqs. 1–3), matches accepted below 2.0 grid units, F1 maximised over rotations (Eq. 4). CHAIR_S / CHAIR_I / CHAIR_instance per Sec. 3.2.

In [ ]:
# ============================== EVALUATE: stage 1 ==============================
FENCE_RE = re.compile(r"```(?:json)?\s*(.*?)```", re.S)
JSON_RE = re.compile(r"\{.*\}", re.S)
TEMPLATE_KEYS = {"category name", "another category"}


def parse_cogmap(text, G, W, H):
    # Returns (map, mode). mode == "grid" means the model obeyed the grid format; that
    # rate is the honest analogue of Table 2's "Succ. Rate(%)". A11: out-of-grid numbers
    # are read as pixels and mapped through Algorithm 1 instead of being discarded, so a
    # formatting failure is not silently reported as a spatial-reasoning failure.
    m = FENCE_RE.search(text)
    body = m.group(1) if m else text
    m = JSON_RE.search(body)
    if not m:
        return None, "no_json"
    blob = m.group(0)
    try:
        obj = json.loads(blob)
    except json.JSONDecodeError:
        try:
            obj = json.loads(re.sub(r",\s*([}\]])", r"\1", blob).replace("'", '"'))
        except json.JSONDecodeError:
            return None, "json_fail"
    if not isinstance(obj, dict):
        return None, "not_dict"
    if {k.lower().strip() for k in obj} & TEMPLATE_KEYS:
        return None, "echoed_template"
    out, modes = {}, set()
    for k, v in obj.items():
        key = normalize_text(k)
        if not key or not isinstance(v, (list, tuple)):
            continue
        cells = []
        for it in v:
            if not isinstance(it, (list, tuple)):
                continue
            try:
                nums = [float(x) for x in it]
            except (TypeError, ValueError):
                continue
            if len(nums) == 4:
                cells.append(grid_cell((nums[0] + nums[2]) / 2, (nums[1] + nums[3]) / 2, W, H, G))
                modes.add("pixel_box")
            elif len(nums) == 2:
                if all(0 <= x < G for x in nums) and all(float(x).is_integer() for x in nums):
                    cells.append((int(nums[0]), int(nums[1])))
                    modes.add("grid")
                else:
                    cells.append(grid_cell(nums[0], nums[1], W, H, G))
                    modes.add("pixel_point")
        if cells:
            out[key] = cells
    if not out:
        return None, "empty"
    mode = "grid" if modes == {"grid"} else ("mixed" if len(modes) > 1 else modes.pop())
    return out, mode


def rotate_cells(cells, k, G):
    # A1: k * 90 degrees about the grid centre.
    out = []
    for (i, j) in cells:
        for _ in range(k % 4):
            i, j = j, G - 1 - i
        out.append((i, j))
    return out


def match_maps(pred, gt, threshold, G):
    # Eqs. 1-3, per object class, with the rotation search of Eq. 4 done by the caller.
    tp, dists = 0, []
    n_pred = sum(len(v) for v in pred.values())
    n_gt = sum(len(v) for v in gt.values())
    for cls, gcells in gt.items():
        pcells = pred.get(cls)
        if not pcells:
            continue
        D = np.zeros((len(gcells), len(pcells)), dtype=np.float64)
        for a, gp in enumerate(gcells):
            for b, pp in enumerate(pcells):
                D[a, b] = math.sqrt((gp[0] - pp[0]) ** 2 + (gp[1] - pp[1]) ** 2)
        rows, cols = linear_sum_assignment(D)
        for a, b in zip(rows, cols):
            if D[a, b] <= threshold:
                tp += 1
                dists.append(float(D[a, b]))
    return tp, n_pred, n_gt, dists


def prf(tp, n_pred, n_gt):
    p = tp / n_pred if n_pred else float("nan")
    r = tp / n_gt if n_gt else float("nan")
    if not n_pred or not n_gt or (p + r) == 0:
        f = float("nan") if (not n_pred or not n_gt) else 0.0
    else:
        f = 2 * p * r / (p + r)
    return p, r, f


def score_cogmap(pred_raw, gt_map, G, threshold, rotations):
    # Eq. 4: evaluate under each rotation, keep the one with the highest F1.
    best = None
    for k in rotations:
        rot = {c: rotate_cells(v, k, G) for c, v in pred_raw.items()}
        tp, n_pred, n_gt, dists = match_maps(rot, gt_map, threshold, G)
        p, r, f = prf(tp, n_pred, n_gt)
        key = -1.0 if (isinstance(f, float) and math.isnan(f)) else f
        if best is None or key > best[0]:
            best = (key, dict(rotation=k, tp=tp, n_pred=n_pred, n_gt=n_gt,
                              precision=p, recall=r, f1=f,
                              dists=dists))
    return best[1]


def chair(pred_raw, gt_pool):
    # Sec. 3.2. CHAIR_S: binary, does the scene contain any hallucinated class.
    # CHAIR_I: hallucinated classes / predicted classes.
    # CHAIR_instance: hallucinated instances / predicted instances.
    gt_named = set(gt_pool)
    pred_classes = list(pred_raw.keys())
    if not pred_classes:
        return float("nan"), float("nan"), float("nan")
    hall_classes = [c for c in pred_classes if c not in gt_named]
    n_inst = sum(len(v) for v in pred_raw.values())
    n_hall_inst = sum(len(pred_raw[c]) for c in hall_classes)
    c_s = 1.0 if hall_classes else 0.0
    c_i = len(hall_classes) / len(pred_classes)
    c_inst = n_hall_inst / n_inst if n_inst else float("nan")
    return c_s, c_i, c_inst


def run_stage1(images, negative):
    tag = "negative" if negative else "clean"
    rows = []
    generated = {}
    t_start = time.time()
    for n, fn in enumerate(images, 1):
        d = image_data[fn]
        cats = list(d["pool"])
        if negative:
            cats = cats + d["negative_classes"]
        if not cats:
            continue
        cats = sorted(set(cats))
        img = load_image(fn)
        raw, _ = generate([("user", cogmap_prompt(cats))], img, CFG.MAX_NEW_TOKENS_COGMAP)
        pred, mode = parse_cogmap(raw, CFG.GRID_G, d["width"], d["height"])
        generated[fn] = raw
        if pred is None:
            pred = {}
        res = score_cogmap(pred, d["cogmap"], CFG.GRID_G, CFG.MATCH_THRESHOLD, CFG.ROTATIONS)
        c_s, c_i, c_inst = chair(pred, d["pool"])
        rows.append(dict(image=fn, condition=tag, mode=mode,
                         success=bool(mode == "grid"), parsed=bool(pred),
                         tp=res["tp"], n_pred=res["n_pred"], n_gt=res["n_gt"],
                         rotation=res["rotation"], dists=res["dists"],
                         chair_s=c_s, chair_i=c_i, chair_instance=c_inst,
                         raw=raw))
        if n % 25 == 0 or n == len(images):
            print(f"  [{tag}] {n}/{len(images)}  {time.time() - t_start:.0f}s", flush=True)
    return rows, generated, time.time() - t_start


def aggregate_stage1(rows):
    tp = sum(r["tp"] for r in rows)
    n_pred = sum(r["n_pred"] for r in rows)
    n_gt = sum(r["n_gt"] for r in rows)
    p, r_, f = prf(tp, n_pred, n_gt)
    all_d = [x for r in rows for x in r["dists"]]
    avg_dist = float(np.mean(all_d)) if all_d else float("nan")
    strict = float(np.mean([1.0 if r["success"] else 0.0 for r in rows])) if rows else float("nan")
    parsed = float(np.mean([1.0 if r["parsed"] else 0.0 for r in rows])) if rows else float("nan")
    modes = dict(Counter(r["mode"] for r in rows))
    def _m(key):
        vals = [r[key] for r in rows if not (isinstance(r[key], float) and math.isnan(r[key]))]
        return float(np.mean(vals)) if vals else float("nan")
    return dict(precision=p, recall=r_, f1=f, avg_dist=avg_dist,
                succ_strict=strict, succ_parsed=parsed, modes=modes,
                chair_s=_m("chair_s"), chair_i=_m("chair_i"),
                chair_instance=_m("chair_instance"), n=len(rows))


stage1_clean_rows, cogmap_by_image, stage1_clean_time = ([], {}, float("nan"))
stage1_clean_agg = None
if CFG.RUN_STAGE1_CLEAN:
    print("stage 1 (clean):")
    stage1_clean_rows, cogmap_by_image, stage1_clean_time = run_stage1(eval_images, negative=False)
    stage1_clean_agg = aggregate_stage1(stage1_clean_rows)
    print("scored images:", stage1_clean_agg["n"],
          "| grid-format rate:", round(stage1_clean_agg["succ_strict"], 4),
          "| parsed rate:", round(stage1_clean_agg["succ_parsed"], 4))
    print("output formats:", stage1_clean_agg["modes"], "| peak GPU GiB:", round(peak_gpu_gib(), 2))

stage1_neg_rows, stage1_neg_time = ([], float("nan"))
stage1_neg_agg = None
if CFG.RUN_STAGE1_NEGATIVE:
    print("stage 1 (negative sampling):")
    stage1_neg_rows, _, stage1_neg_time = run_stage1(eval_images, negative=True)
    stage1_neg_agg = aggregate_stage1(stage1_neg_rows)
    print("scored images:", stage1_neg_agg["n"],
          "| grid-format rate:", round(stage1_neg_agg["succ_strict"], 4))
    print("output formats:", stage1_neg_agg["modes"], "| peak GPU GiB:", round(peak_gpu_gib(), 2))

with open(os.path.join(CFG.OUT_DIR, "stage1_predictions.jsonl"), "w") as f:
    for r in stage1_clean_rows + stage1_neg_rows:
        f.write(json.dumps({k: v for k, v in r.items() if k != "dists"}) + "\n")
print("wrote stage1_predictions.jsonl")

## EVALUATE — stage 2: question answering

Round two of the dialogue. `w/ cogmap` replays the model's own stage-1 map as the assistant turn; `w/o cogmap` asks the question directly (Table 3). Counting uses the proportional score (Sec. 3.2, formula assumed — A2); everything else is binary regex matching.

In [ ]:
# ============================== EVALUATE: stage 2 ==============================
ANSWER_TAG_RE = re.compile(r"<answer>(.*?)</answer>", re.S | re.I)
INT_RE = re.compile(r"-?\d+")
ABSENCE_RE = re.compile(
    r"\b(0|no|none|not found|not present|there is no|there are no|does not|"
    r"doesn t|isn t|aren t|cannot find|can t find|absent)\b")


def extract_answer(raw):
    m = ANSWER_TAG_RE.search(raw)
    return (m.group(1) if m else raw).strip()


def regex_match(pred_raw, gt):
    # Sec. 3.2: "we employ regex pattern matching to evaluate model responses
    # against ground truth answers".
    p = normalize_text(extract_answer(pred_raw))
    g = normalize_text(gt)
    if not g:
        return 0.0
    if p == g:
        return 1.0
    return 1.0 if re.search(r"\b" + re.escape(g) + r"\b", p) else 0.0


def extract_count(pred_raw):
    # The tagged span is preferred, but a reply like "<answer>none</answer> There are no
    # bicycles" carries the decision outside the tag too, so fall back to the full text
    # before giving up. 67 of 750 negative-sampling replies were unparsed without this.
    for text in (extract_answer(pred_raw), pred_raw):
        p = normalize_text(text)
        m = INT_RE.search(p)
        if m:
            return int(m.group(0))
        if ABSENCE_RE.search(p):
            return 0
    return None


def count_score(pred_raw, gt_answer):
    # A2: paper says 100% on exact match, "decreasing proportionally as the numerical
    # difference increases", but gives no formula.
    g = normalize_text(gt_answer)
    m = INT_RE.search(g)
    if not m:
        return regex_match(pred_raw, gt_answer), None, None
    gt = int(m.group(0))
    pred = extract_count(pred_raw)
    if pred is None:
        return 0.0, None, gt
    s = max(0.0, 1.0 - abs(pred - gt) / max(abs(gt), 1))
    return s, pred, gt


def score_record(pred_raw, rec):
    cat = question_category(rec["question"])
    if cat == "object_count":
        s, pred_n, gt_n = count_score(pred_raw, rec["answer"])
        return cat, s, pred_n, gt_n
    return cat, regex_match(pred_raw, rec["answer"]), None, None


def run_stage2(recs, use_cogmap):
    tag = "w_cogmap" if use_cogmap else "wo_cogmap"
    rows, t_start = [], time.time()
    for n, rec in enumerate(recs, 1):
        fn = rec["image"]
        if use_cogmap:
            prior = cogmap_by_image.get(fn)
            if prior is None:
                continue
            d = image_data[fn]
            cats = sorted({c for c in d["pool"] if c != "object"})
            turns = [("user", cogmap_prompt(cats)),
                     ("assistant", prior),
                     ("user", qa_prompt(rec["question"]))]
        else:
            turns = [("user", qa_prompt(rec["question"]))]
        img = load_image(fn)
        raw, _ = generate(turns, img, CFG.MAX_NEW_TOKENS_QA)
        cat, s, pred_n, gt_n = score_record(raw, rec)
        rows.append(dict(image=fn, condition=tag, category=cat, score=float(s),
                         question=rec["question"], gt=rec["answer"], pred=raw,
                         pred_count=pred_n, gt_count=gt_n))
        if n % 50 == 0 or n == len(recs):
            print(f"  [{tag}] {n}/{len(recs)}  {time.time() - t_start:.0f}s", flush=True)
    return rows, time.time() - t_start


def run_stage2_negative(images):
    # Fig. 2: a negative-sampling counting question has ground truth 0.
    rows, t_start = [], time.time()
    for n, fn in enumerate(images, 1):
        d = image_data[fn]
        for cls in d["negative_classes"]:
            q = negative_count_question(cls)
            img = load_image(fn)
            raw, _ = generate([("user", qa_prompt(q))], img, CFG.MAX_NEW_TOKENS_QA)
            s, pred_n, gt_n = count_score(raw, "0")
            rows.append(dict(image=fn, condition="negative", category="object_count",
                             score=float(s), question=q, gt="0", pred=raw,
                             pred_count=pred_n, gt_count=0, injected_class=cls))
        if n % 25 == 0 or n == len(images):
            print(f"  [negative] {n}/{len(images)}  {time.time() - t_start:.0f}s", flush=True)
    return rows, time.time() - t_start


def aggregate_stage2(rows):
    if not rows:
        return dict(acc=float("nan"), n=0, by_cat={})
    acc = float(np.mean([r["score"] for r in rows]))
    by_cat = {}
    for cat in sorted({r["category"] for r in rows}):
        vals = [r["score"] for r in rows if r["category"] == cat]
        by_cat[cat] = float(np.mean(vals))
    return dict(acc=acc, n=len(rows), by_cat=by_cat)


s2_with_rows, s2_with_time = ([], float("nan"))
if CFG.RUN_STAGE2_WITH_COGMAP and cogmap_by_image:
    print("stage 2 (w/ cogmap):")
    s2_with_rows, s2_with_time = run_stage2(eval_records, use_cogmap=True)
s2_with_agg = aggregate_stage2(s2_with_rows)

s2_without_rows, s2_without_time = ([], float("nan"))
if CFG.RUN_STAGE2_WITHOUT_COGMAP:
    print("stage 2 (w/o cogmap):")
    s2_without_rows, s2_without_time = run_stage2(eval_records, use_cogmap=False)
s2_without_agg = aggregate_stage2(s2_without_rows)

# A12: the two conditions must be compared on the same records. Stage 2 w/ cogmap skips
# images with no stage-1 map, so the unpaired accuracies answer different questions.
_w = {(r["image"], r["question"]): r for r in s2_with_rows}
_wo = {(r["image"], r["question"]): r for r in s2_without_rows}
paired_keys = sorted(set(_w) & set(_wo))
if paired_keys:
    paired_w = float(np.mean([_w[k]["score"] for k in paired_keys]))
    paired_wo = float(np.mean([_wo[k]["score"] for k in paired_keys]))
    cogmap_wins = sum(1 for k in paired_keys if _w[k]["score"] > _wo[k]["score"])
    cogmap_losses = sum(1 for k in paired_keys if _w[k]["score"] < _wo[k]["score"])
    paired_w_cat = {c: float(np.mean([_w[k]["score"] for k in paired_keys
                                      if _w[k]["category"] == c]))
                    for c in sorted({_w[k]["category"] for k in paired_keys})}
    paired_wo_cat = {c: float(np.mean([_wo[k]["score"] for k in paired_keys
                                       if _wo[k]["category"] == c]))
                     for c in sorted({_wo[k]["category"] for k in paired_keys})}
    print(f"paired on {len(paired_keys)} records: w/ cogmap={paired_w:.4f} "
          f"w/o cogmap={paired_wo:.4f} delta={paired_w - paired_wo:+.4f}")
    print(f"  discordant: cogmap wins {cogmap_wins}, loses {cogmap_losses}, "
          f"ties {len(paired_keys) - cogmap_wins - cogmap_losses}")
    print("  w/ by category :", paired_w_cat)
    print("  w/o by category:", paired_wo_cat)
else:
    paired_w = paired_wo = float("nan")
    cogmap_wins = cogmap_losses = 0
    paired_w_cat = paired_wo_cat = {}

# The paper's object-counting category has no counterpart among real questions here.
_n_real_count = sum(1 for r in eval_records if question_category(r["question"]) == "object_count")
print("real object-counting questions in the subset:", _n_real_count)
if _n_real_count == 0:
    print("  -> obj_count_score is reported only for synthetic negative-sampling questions.")

s2_neg_rows, s2_neg_time = ([], float("nan"))
if CFG.RUN_STAGE2_NEGATIVE:
    print("stage 2 (negative sampling):")
    s2_neg_rows, s2_neg_time = run_stage2_negative(eval_images)
s2_neg_agg = aggregate_stage2(s2_neg_rows)

# --- FPR / FNR on the presence decision -------------------------------------------
# Positive class = "the asked-about object is present". Negatives are the injected
# absent classes; positives are counting questions whose ground-truth count >= 1.
# Micro-averaged over questions. The paper does not report FPR/FNR.
n_contaminated = sum(1 for r in s2_neg_rows
                     if r["injected_class"] in image_data[r["image"]]["cogmap"])
print("injected distractors that are annotated present:", n_contaminated,
      f"of {len(s2_neg_rows)}")
neg_decisions = [r["pred_count"] for r in s2_neg_rows if r["pred_count"] is not None]
pos_rows = [r for r in s2_without_rows
            if r["category"] == "object_count" and r["gt_count"] is not None
            and r["gt_count"] >= 1 and r["pred_count"] is not None]
fp = sum(1 for c in neg_decisions if c >= 1)
fn_ = sum(1 for r in pos_rows if r["pred_count"] == 0)
FPR = fp / len(neg_decisions) if neg_decisions else float("nan")
FNR = fn_ / len(pos_rows) if pos_rows else float("nan")
n_unparsed_neg = len(s2_neg_rows) - len(neg_decisions)
print("presence decisions - negatives:", len(neg_decisions),
      "positives:", len(pos_rows), "unparsed negatives:", n_unparsed_neg)

with open(os.path.join(CFG.OUT_DIR, "stage2_predictions.jsonl"), "w") as f:
    for r in s2_with_rows + s2_without_rows + s2_neg_rows:
        f.write(json.dumps(r) + "\n")
print("wrote stage2_predictions.jsonl")

## EVALUATE — visual grounding (adaptation, not in the paper)

The paper has no bounding-box grounding task. This section exists because the mounted annotations carry pixel boxes and the setup asks for grounding accuracy with and without full-frame boxes. Reported as a separate row, never mixed into the paper's metrics.

In [ ]:
# ============================== EVALUATE: grounding ==============================
BOX_RE = re.compile(r"\[\s*(-?\d+\.?\d*)\s*,\s*(-?\d+\.?\d*)\s*,"
                    r"\s*(-?\d+\.?\d*)\s*,\s*(-?\d+\.?\d*)\s*\]")


def parse_box(text):
    m = BOX_RE.search(text)
    if not m:
        return None
    return [float(m.group(i)) for i in range(1, 5)]


def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    ua = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    ub = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    denom = ua + ub - inter
    return inter / denom if denom > 0 else 0.0


def run_grounding(recs):
    rows, t_start = [], time.time()
    for n, rec in enumerate(recs, 1):
        fn = rec["image"]
        W, H = int(rec["width"]), int(rec["height"])
        img = load_image(fn)
        raw, resized_hw = generate([("user", grounding_prompt(rec["question"]))],
                                   img, CFG.MAX_NEW_TOKENS_BOX)
        box = parse_box(raw)
        best = float("nan")
        if box is not None and resized_hw is not None:
            # Qwen2.5-VL returns pixels of the smart-resized image; rescale to original.
            rh, rw = resized_hw
            sx, sy = W / float(rw), H / float(rh)
            box = [box[0] * sx, box[1] * sy, box[2] * sx, box[3] * sy]
            best = max(iou(box, g) for g in rec["bboxs_clamped"])
        elif box is not None:
            best = max(iou(box, g) for g in rec["bboxs_clamped"])
        rows.append(dict(image=fn, question=rec["question"], pred=raw,
                         iou=float(best) if box is not None else float("nan"),
                         parsed=box is not None,
                         has_fullframe=bool(rec["has_fullframe_box"])))
        if n % 50 == 0 or n == len(recs):
            print(f"  [grounding] {n}/{len(recs)}  {time.time() - t_start:.0f}s", flush=True)
    return rows, time.time() - t_start


ground_rows, ground_time = ([], float("nan"))
if CFG.RUN_GROUNDING:
    print("grounding:")
    ground_rows, ground_time = run_grounding(eval_records)


def grounding_acc(rows, thr):
    if not rows:
        return float("nan")
    hits = [1.0 if (r["parsed"] and not math.isnan(r["iou"]) and r["iou"] >= thr) else 0.0
            for r in rows]
    return float(np.mean(hits))


ground_all = grounding_acc(ground_rows, CFG.IOU_THRESHOLD)
ground_excl = grounding_acc([r for r in ground_rows if not r["has_fullframe"]],
                            CFG.IOU_THRESHOLD)
n_ground_excl = len([r for r in ground_rows if not r["has_fullframe"]])
parse_rate = (float(np.mean([1.0 if r["parsed"] else 0.0 for r in ground_rows]))
              if ground_rows else float("nan"))
print("grounding rows:", len(ground_rows), "| excluding full-frame:", n_ground_excl)

with open(os.path.join(CFG.OUT_DIR, "grounding_predictions.jsonl"), "w") as f:
    for r in ground_rows:
        f.write(json.dumps(r) + "\n")
print("wrote grounding_predictions.jsonl")

## SAVE RESULTS

Every cell traces back to a variable computed above. `nan` becomes the string `N/A` only at write time.

In [ ]:
# ============================== SAVE RESULTS ==============================
PEAK_GIB = peak_gpu_gib()
INPUTS = f"{CFG.ANN_PATH} | {CFG.IMAGE_ROOT}"
SOURCE = "Dongfang et al., Are MLLMs Ready for Omnidirectional Spatial Reasoning? (CVPR Findings 2026)"
SPLIT = "gqa_cot_val.jsonl (val)"
FIDELITY = "adaptation"
NA = float("nan")

rows_out = []


def base_row(exp, n_eval, inference_time, extra):
    row = {
        "experiment": exp,
        "Acc": NA, "Prec": NA, "Recall": NA, "F1": NA,
        "ROC-AUC": NA, "PR-AUC": NA, "FPR": NA, "FNR": NA,
        "Train Time": train_time_s, "Params": PARAM_COUNT,
        "Comm Cost": NA, "Training Steps": n_training_steps,
        "n_eval": n_eval, "source": SOURCE, "split": SPLIT,
        "model": CFG.MODEL_SHORT, "inputs": INPUTS, "fidelity": FIDELITY,
        "inference_time_s": inference_time, "peak_gpu_gib": PEAK_GIB,
        "avg_dist": NA, "succ_rate_strict_grid": NA, "succ_rate_parsed": NA,
        "chair_s": NA, "chair_i": NA, "chair_instance": NA,
        "obj_count_score": NA, "spatial_relation_score": NA, "other_score": NA,
        "grounding_acc_excl_fullframe": NA, "grounding_parse_rate": NA, "notes": "",
    }
    row.update(extra)
    return row


if stage1_clean_agg is not None:
    a = stage1_clean_agg
    rows_out.append(base_row(
        "stage1_cogmap_clean", a["n"], stage1_clean_time,
        {"Prec": a["precision"], "Recall": a["recall"], "F1": a["f1"],
         "avg_dist": a["avg_dist"], "succ_rate_strict_grid": a["succ_strict"],
         "succ_rate_parsed": a["succ_parsed"],
         "chair_s": a["chair_s"], "chair_i": a["chair_i"],
         "chair_instance": a["chair_instance"],
         "notes": "F1 uses coordinate-tolerant parsing (A11); "
                  "succ_rate_strict_grid is the Table-2 analogue"}))

if stage1_neg_agg is not None:
    a = stage1_neg_agg
    rows_out.append(base_row(
        "stage1_cogmap_negative_sampling", a["n"], stage1_neg_time,
        {"Prec": a["precision"], "Recall": a["recall"], "F1": a["f1"],
         "avg_dist": a["avg_dist"], "succ_rate_strict_grid": a["succ_strict"],
         "succ_rate_parsed": a["succ_parsed"],
         "chair_s": a["chair_s"], "chair_i": a["chair_i"],
         "chair_instance": a["chair_instance"],
         "notes": "CHAIR is an upper bound: GT pools annotate only the referents"}))

if paired_keys:
    rows_out.append(base_row(
        "stage2_qa_w_cogmap_paired", len(paired_keys), s2_with_time,
        {"Acc": paired_w,
         "spatial_relation_score": paired_w_cat.get("spatial_relation", NA),
         "other_score": paired_w_cat.get("other", NA),
         "notes": f"paired with wo_cogmap; cogmap wins {cogmap_wins}, "
                  f"loses {cogmap_losses}"}))
    rows_out.append(base_row(
        "stage2_qa_wo_cogmap_paired", len(paired_keys), NA,
        {"Acc": paired_wo,
         "spatial_relation_score": paired_wo_cat.get("spatial_relation", NA),
         "other_score": paired_wo_cat.get("other", NA),
         "notes": "subset of the full wo_cogmap run; wall time not separately measured"}))

if s2_without_rows:
    a = s2_without_agg
    rows_out.append(base_row(
        "stage2_qa_wo_cogmap_all", a["n"], s2_without_time,
        {"Acc": a["acc"], "FNR": FNR,
         "obj_count_score": a["by_cat"].get("object_count", NA),
         "spatial_relation_score": a["by_cat"].get("spatial_relation", NA),
         "other_score": a["by_cat"].get("other", NA),
         "notes": "not comparable to w_cogmap: different record set"}))

if s2_neg_rows:
    a = s2_neg_agg
    rows_out.append(base_row(
        "stage2_qa_negative_sampling", a["n"], s2_neg_time,
        {"Acc": a["acc"], "FPR": FPR,
         "obj_count_score": a["by_cat"].get("object_count", NA),
         "notes": f"synthetic counting questions only; {n_contaminated} injected "
                  "distractors are annotated present"}))

if ground_rows:
    rows_out.append(base_row(
        "grounding_bbox_iou50_adaptation", len(ground_rows), ground_time,
        {"Acc": ground_all,
         "grounding_acc_excl_fullframe": ground_excl,
         "grounding_parse_rate": parse_rate,
         "notes": "not a paper metric; boxes rescaled from the smart-resized space"}))

if not rows_out:
    raise RuntimeError("No experiment produced results. Check the CFG.RUN_* switches.")

COLS = ["Acc", "Prec", "Recall", "F1", "ROC-AUC", "PR-AUC", "FPR", "FNR",
        "Train Time", "Params", "Comm Cost", "Training Steps",
        "n_eval", "source", "split", "model", "inputs", "fidelity",
        "experiment", "inference_time_s", "peak_gpu_gib",
        "avg_dist", "succ_rate_strict_grid", "succ_rate_parsed",
        "chair_s", "chair_i", "chair_instance",
        "obj_count_score", "spatial_relation_score", "other_score",
        "grounding_acc_excl_fullframe", "grounding_parse_rate", "notes"]

results = pd.DataFrame(rows_out)[COLS]


def to_cell(v):
    if isinstance(v, float) and math.isnan(v):
        return "N/A"
    return v


results_out = results.copy()
for _c in results.columns:
    results_out[_c] = results[_c].map(to_cell)
results_out.to_csv(os.path.join(CFG.OUT_DIR, "results.csv"), index=False)

pd.set_option("display.max_columns", None, "display.width", 250)
display(results_out)

print()
print("ROC-AUC and PR-AUC are N/A: answers are short strings scored by normalized")
print("regex/exact match, and there is no label-independent score. Building one out of")
print("correctness would return 1.0 by construction.")
print("Comm Cost is N/A: the paper measures no communication.")
print("Training Steps is 0 and Train Time is the measured wall time of a no-op section:")
print("the paper is zero-shot (Sec. 3.3).")
print("Prec/Recall/F1 are the stage-1 cognitive-map metrics (Eqs. 1-4), micro-averaged")
print("over all matched instances; the paper does not state its averaging.")
print("FPR/FNR use positive class = 'object is present', micro-averaged over counting")
print("questions; the paper does not report them.")
print()
print("wrote", os.path.join(CFG.OUT_DIR, "results.csv"))